# Tema 3: ¿resisten los resultados probar muchas estrategias?

Notebook independiente, sin conexión a la terminal, cartera, claves ni proveedores.
Referencia: exceso medio neto **0** frente a un benchmark emparejado por fecha.
H0: exceso medio ≤ 0; H1: > 0. El ejemplo predeterminado es **SINTÉTICO**:
demuestra ejecución, bloquea conclusiones financieras y no representa mercado.

Antes de cargar datos reales: declare **todas** las variantes ensayadas, costos,
fuente, regla de selección y cortes; guarde externamente la configuración y su huella
con fecha verificable. No vea el holdout hasta congelar la selección. Los rendimientos
deben proceder de estrategias causales, estimadas fuera de muestra mediante walk-forward.
La separación aquí no reconstruye ni certifica ese proceso.


## 1. Instalación explícita y versiones
En Colab, cambie el interruptor a `True` solo para instalar estas dos bibliotecas gratuitas.
Reinicie la sesión si Colab lo solicita y después vuelva a `False`.
La prueba local ejecuta todas las celdas sin instalar ni descargar nada.


In [ ]:
INSTALAR_DEPENDENCIAS = False
VERSIONES = {"numpy": "2.5.3", "pandas": "3.0.6"}
if INSTALAR_DEPENDENCIAS:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install",
                           *[f"{k}=={v}" for k, v in VERSIONES.items()]])
import csv, hashlib, io, json, platform
from pathlib import Path
import numpy as np
import pandas as pd
assert {"numpy": np.__version__, "pandas": pd.__version__} == VERSIONES, "Instale las versiones fijadas."
print({"python": platform.python_version(), **VERSIONES})


## 2. Configuración y declaración previa
En modo real use un CSV local: `fecha,benchmark,variante_1,variante_2,variante_3`.
Fechas ISO únicas crecientes; rendimientos simples netos en decimal (0.01 = 1 %), misma
moneda, periodo y sesiones, sin rellenar faltantes. Incluya comisiones, impuestos aplicables,
deslizamiento y financiamiento. El benchmark también debe ser neto y comparable.
`familia_total` cuenta además variantes no publicadas, tratadas conservadoramente como p=1.
No reduzca esta familia después de observar resultados. Configure la purga al horizonte
máximo de etiquetas/posiciones solapadas y el embargo a la dependencia que deba excluir.


In [ ]:
CONFIG = {
    "modo": "SINTÉTICO", "csv": "rendimientos_netos.csv", "semilla": 20261016,
    "estrategias": ["variante_1", "variante_2", "variante_3"], "familia_total": 3,
    "B": 4999, "bloque": 5, "alpha": 0.05, "holdout": 0.30,
    "purga": 5, "embargo": 5,
    "seleccion": "mayor exceso medio en exploración; empate por orden declarado",
    "fuente": "ejemplo sintético", "costos": "no representa costos reales",
    "generacion_causal": "ejemplo sin señales ni ajuste de estrategias",
    "confirmo_netos_y_emparejados": False, "confirmo_preregistro_externo": False,
}
assert CONFIG["modo"] in {"SINTÉTICO", "REAL"}
assert 1 <= len(CONFIG["estrategias"]) <= 20
assert len(set(CONFIG["estrategias"])) == len(CONFIG["estrategias"])
assert not ({"fecha", "benchmark"} & set(CONFIG["estrategias"]))
assert all(isinstance(n, str) and n.strip() and len(n) <= 40 for n in CONFIG["estrategias"])
assert len(CONFIG["estrategias"]) <= CONFIG["familia_total"] <= 1000
assert 199 <= CONFIG["B"] <= 20000 and 1 <= CONFIG["bloque"] <= 250
assert 0 < CONFIG["alpha"] < 0.5 and 0.1 <= CONFIG["holdout"] <= 0.5
assert 0 <= CONFIG["purga"] <= 250 and 0 <= CONFIG["embargo"] <= 250
if CONFIG["modo"] == "REAL":
    assert CONFIG["confirmo_netos_y_emparejados"] and CONFIG["confirmo_preregistro_externo"], "Confirme netos y preregistro externo."
    assert all(CONFIG[k].strip() and "ejemplo" not in CONFIG[k].lower()
               for k in ("fuente", "costos", "generacion_causal")), "Documente fuente, costos y generación causal reales."
config_congelada = json.dumps(CONFIG, ensure_ascii=False, sort_keys=True)
huella_config = hashlib.sha256(config_congelada.encode()).hexdigest()
print("Guarde ANTES de cargar datos:", config_congelada, "SHA256:", huella_config)


## 3. Datos locales o ejemplo SINTÉTICO
El CSV se carga manualmente al panel Archivos de Colab. No hay descargas de mercado.
No se inspecciona ni grafica el tramo reservado. La integridad global se valida sin
mostrar valores de holdout. Todas las variantes se generan bajo exceso esperado cero en
el ejemplo; una aparente ganadora puede surgir solo por azar.


In [ ]:
if CONFIG["modo"] == "SINTÉTICO":
    rng = np.random.default_rng(CONFIG["semilla"])
    benchmark = rng.normal(0.0002, 0.009, 600)
    datos = pd.DataFrame({"fecha": pd.bdate_range("2022-01-03", periods=600), "benchmark": benchmark})
    for nombre in CONFIG["estrategias"]:
        datos[nombre] = benchmark + rng.normal(0, 0.004, len(datos))
    contenido = datos.to_csv(index=False).encode()
else:
    ruta = Path(CONFIG["csv"])
    assert ruta.is_file() and ruta.stat().st_size <= 1_000_000, "CSV local requerido, máximo 1 MB."
    contenido = ruta.read_bytes()
    encabezado = next(csv.reader(io.StringIO(contenido.decode("utf-8-sig"))))
    assert len(encabezado) == len(set(encabezado)), "Encabezados duplicados."
    datos = pd.read_csv(io.BytesIO(contenido))
huella_datos = hashlib.sha256(contenido).hexdigest()
print(CONFIG["modo"], "filas:", len(datos), "SHA256:", huella_datos)


## 4. Validación y separación cronológica
Exploración | purga descartada | embargo descartado | holdout reservado.
Purga y embargo se cuentan en observaciones. No se ordena, imputa ni alinea silenciosamente.
Cada tramo usado necesita al menos ocho bloques; un bloque final parcial se conserva.


In [ ]:
def validar_y_separar(tabla, config):
    columnas = ["fecha", "benchmark", *config["estrategias"]]
    assert list(tabla.columns) == columnas, "Declare exactamente todas las columnas y su orden."
    assert 100 <= len(tabla) <= 5000, "Se necesitan entre 100 y 5000 observaciones."
    fechas = pd.to_datetime(tabla["fecha"], format="ISO8601", errors="raise")
    assert fechas.notna().all() and fechas.is_unique and fechas.is_monotonic_increasing, "Fechas únicas crecientes."
    valores = tabla[columnas[1:]]
    assert not any(valores[c].map(lambda x: isinstance(x, (bool, np.bool_))).any() for c in valores), "Booleanos inválidos."
    numeros = valores.apply(pd.to_numeric, errors="raise").to_numpy(dtype=float)
    assert np.isfinite(numeros).all() and (numeros > -1).all() and (numeros <= 10).all(), "Rendimientos inválidos o no decimales."
    nh = int(len(tabla) * config["holdout"])
    inicio_holdout = len(tabla) - nh
    fin_exploracion = inicio_holdout - config["purga"] - config["embargo"]
    assert min(fin_exploracion, nh) >= 8 * config["bloque"], "Muy pocos bloques."
    assert config["B"] * int(np.ceil(len(tabla) / config["bloque"])) * len(config["estrategias"]) <= 200_000_000, "Cómputo excesivo."
    return tabla.iloc[:fin_exploracion].copy(), tabla.iloc[inicio_holdout:].copy(), {
        "exploracion": fin_exploracion, "purga": config["purga"], "embargo": config["embargo"], "holdout": nh,
        "ultima_fecha_exploracion": str(fechas.iloc[fin_exploracion - 1]),
        "primera_fecha_holdout": str(fechas.iloc[inicio_holdout]),
    }
exploracion, reservado, tramos = validar_y_separar(datos, CONFIG)
print(tramos)


## 5. Análisis: selección solo en exploración
La selección ignora el holdout. No se ajustan reglas ni parámetros aquí.
La tabla y sus excesos no prueban ventaja; probar más variantes aumenta la selección por azar.


In [ ]:
excesos_exploracion = exploracion[CONFIG["estrategias"]].sub(exploracion["benchmark"], axis=0)
medias = excesos_exploracion.mean()
seleccionada = str(medias.idxmax())
print(CONFIG["modo"], "selección congelada:", seleccionada)
print(medias.rename("exceso_medio_exploracion").to_string())
seleccion_congelada = seleccionada


## 6. Evaluación: holdout y multiplicidad
Volteo conjunto de signos de bloques contiguos; p=(extremos+1)/(B+1).
Holm corrige **toda** la familia declarada; no solo la ganadora. La decisión principal
corresponde a la variante elegida en exploración. Cada fila, incluidos los resultados nulos,
se conserva en la exportación. Se supone simetría/intercambiabilidad de signos bajo H0;
colas, heterocedasticidad o dependencia mayor que el bloque pueden invalidar la inferencia.
Un no rechazo no demuestra equivalencia ni ausencia de efecto.

Reejecutar esta celda devuelve el resultado congelado en esta sesión. Alterar configuración,
datos o selección lo bloquea. **No es un servicio de preregistro**: reiniciar la sesión o
editar el código elude este control. Guarde el resultado y no reutilice el holdout; reserve
datos futuros para un experimento nuevo. No interprete la simulación como evidencia real.


In [ ]:
def evaluar_familia(tramo, config):
    d = tramo[config["estrategias"]].sub(tramo["benchmark"], axis=0).to_numpy(dtype=float)
    sumas = np.add.reduceat(d, np.arange(0, len(d), config["bloque"]), axis=0)
    observadas = d.mean(axis=0)
    rng = np.random.default_rng(config["semilla"])
    extremos = np.zeros(d.shape[1], dtype=int)
    for inicio in range(0, config["B"], 500):
        signos = rng.integers(0, 2, size=(min(500, config["B"] - inicio), len(sumas))) * 2 - 1
        extremos += ((signos @ sumas) / len(d) >= observadas - 1e-12 * max(float(abs(d).max()), 1e-300)).sum(axis=0)
    p = (extremos + 1) / (config["B"] + 1)
    orden = np.argsort(p, kind="stable")
    ajustados = np.empty(len(p))
    ajustados[orden] = np.minimum(1, np.maximum.accumulate(p[orden] * (config["familia_total"] - np.arange(len(p)))))
    return pd.DataFrame({"estrategia": config["estrategias"], "exceso_medio": observadas,
                         "p": p, "p_holm": ajustados})

assert json.dumps(CONFIG, ensure_ascii=False, sort_keys=True) == config_congelada, "Configuración alterada."
assert seleccionada == seleccion_congelada, "Selección alterada."
firma_evaluacion = hashlib.sha256((config_congelada + datos.to_csv(index=False) + seleccion_congelada).encode()).hexdigest()
if "resultado_congelado" in globals():
    assert resultado_congelado["firma_evaluacion"] == firma_evaluacion, "Evaluación ya consumida: no cambie datos ni selección."
else:
    # Reextraer los tramos impide evaluar una copia de holdout manipulada accidentalmente.
    _, holdout_final, _ = validar_y_separar(datos, CONFIG)
    tabla_final = evaluar_familia(holdout_final, CONFIG)
    principal = tabla_final.loc[tabla_final.estrategia == seleccion_congelada].iloc[0]
    conclusion = "BLOQUEADA: SINTÉTICO; ninguna conclusión financiera."
    if CONFIG["modo"] == "REAL":
        conclusion = ("Evidencia condicional de exceso neto positivo; no demuestra rentabilidad futura."
                      if principal.p_holm <= CONFIG["alpha"] else "No se rechaza H0 tras Holm: evidencia insuficiente.")
    resultado_congelado = {
        "modo": CONFIG["modo"], "conclusion": conclusion, "seleccionada": seleccion_congelada,
        "firma_evaluacion": firma_evaluacion, "huella_config": huella_config, "huella_csv": huella_datos,
        "configuracion": json.loads(config_congelada), "versiones": {"python": platform.python_version(), **VERSIONES},
        "tramos": tramos, "referencia_exceso": 0, "p_minimo": 1 / (CONFIG["B"] + 1),
        "resultados_holdout": tabla_final.to_dict(orient="records"),
    }
print(resultado_congelado["conclusion"])
print(pd.DataFrame(resultado_congelado["resultados_holdout"]).to_string(index=False))


## 7. Exportación local
JSON completo y CSV de todas las variantes. El nombre incorpora la huella del experimento;
no sobrescribe resultados anteriores. Descárguelos desde Archivos de Colab.
No publica ni envía nada. Las fechas sintéticas son solo un calendario del ejemplo.


In [ ]:
SALIDA = Path("resultados_tema3")
SALIDA.mkdir(parents=True, exist_ok=True)
prefijo = f"{resultado_congelado['modo']}_{resultado_congelado['firma_evaluacion'][:16]}"
artefactos = {
    SALIDA / f"{prefijo}.json": json.dumps(resultado_congelado, ensure_ascii=False, indent=2, allow_nan=False),
    SALIDA / f"{prefijo}.csv": pd.DataFrame(resultado_congelado["resultados_holdout"]).to_csv(index=False, lineterminator="\n"),
}
for ruta, texto in artefactos.items():
    if ruta.exists():
        assert ruta.read_text(encoding="utf-8") == texto, "Archivo previo distinto: no se sobrescribe."
    else:
        with ruta.open("x", encoding="utf-8", newline="") as archivo:
            archivo.write(texto)
print("Archivos locales:", *map(str, artefactos))


## Interpretación y límites
Ejemplo manual de Holm: p=[0.01, 0.04, 0.20], familia=3 → [0.03, 0.08, 0.20].
Si probó más variantes que las declaradas, la corrección es incompleta. Aumentar B mejora
resolución Monte Carlo, no calidad de datos. Los costos, derechos de uso y causalidad del
CSV los acredita el usuario. No hay cálculo de señales, conexión a bróker ni backtest nuevo.
En caso de solapamiento más largo o dependencia fuerte, amplíe la separación antes de
ver datos y considere un diseño distinto; nunca ajuste bloques viendo el holdout.
No se afirma ventaja predictiva ni rentabilidad con este notebook.
